# Arrow数据类型与互操作

学习目标：使用 Arrow 支持的 pandas 类型处理缺失值，区分解析引擎与存储后端，检查字符串语义和 NumPy 转换；选学 Arrow Table 往返与共享条件。

前置知识：扩展 dtype、缺失值、格式读写、内存估计。

运行环境：Python 3.12、pandas 3、PyArrow 25；字符串示例按 pandas 3 的默认 str 规则编写。

环境准备：[环境配置与运行](README.md)

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章为扩展内容，使用内存中的自制小表，不读取外部数据。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 保留整数和缺失值

下面的库存 CSV 有一条数量尚未确认。读取时指定 dtype_backend="pyarrow"，让数量列使用 Arrow 支持的可空整数类型；空字段保留为缺失，不必先变成浮点数。

Arrow 是用于数据交换与计算的列式数据格式，PyArrow 是其 Python 接口。它也可以作为 pandas 列的底层存储，读者仍然使用 DataFrame、Series 和常见 pandas 操作。

In [1]:
from io import StringIO

import numpy as np
import pandas as pd
import pyarrow as pa

csv_text = "item,qty\napple,2\npear,\nplum,5\n"
with StringIO(csv_text) as source:
    stock = pd.read_csv(source, engine="c", dtype_backend="pyarrow")
print(stock)  # 预期：item 为 apple/pear/plum，qty 为 2/<NA>/5。
print(stock.dtypes)  # 预期：item 为 string[pyarrow]，qty 为 int64[pyarrow]。
print(stock.loc[stock["qty"].isna(), "item"])
print(stock["qty"].sum())
# qty 是 int64[pyarrow]；pear 的数量为 <NA>，已知数量合计为 7 件。
# sum 默认跳过缺失，7 只表示已知总量，不能据此把 pear 的数量当作 0。

    item   qty
0  apple     2
1   pear  <NA>
2   plum     5
item    string[pyarrow]
qty      int64[pyarrow]
dtype: object
1    pear
Name: item, dtype: string[pyarrow]
7


## 2 ArrowDtype 与单列类型

ArrowDtype 是 pandas 的一种扩展 dtype，用来描述 PyArrow 类型。简单类型可以写成 "int64[pyarrow]"，也可以显式写 pd.ArrowDtype(pa.int64())。

Series 的标签仍由 pandas 管理。底层列可以采用 Arrow 的 ChunkedArray，即由一个或多个同类型片段组成的数组；它不是 NumPy ndarray。需要 pandas 扩展数组时用 array，不必先转成 NumPy。

In [2]:
counts = pd.Series(
    [2, None, 5], index=["A", "B", "C"], dtype=pd.ArrowDtype(pa.int64()),
)
print(counts)  # 预期：A=2、B=<NA>、C=5，dtype 为 int64[pyarrow]。
print(counts.dtype == pd.ArrowDtype(pa.int64()))  # 预期：True。
print(type(counts.array).__name__)
print(pa.array(counts))
# 标签 A、B、C 保留；array 是 ArrowExtensionArray，转为 Arrow 后缺失显示为 null。

A       2
B    <NA>
C       5
dtype: int64[pyarrow]
True
ArrowExtensionArray
[
  2,
  null,
  5
]


## 3 转换整张表的后端

dtype_backend 是转换或读取时的一项选择，不是某一种具体列类型。convert_dtypes 可以把已有表转为支持缺失值的类型："numpy_nullable" 选择 pandas 可空类型，"pyarrow" 选择 ArrowDtype。

这不是数据清洗规则，也不会给缺失值补出事实。转换后仍须检查值、列名、索引和 dtype。"numpy_nullable" 也不保证每一列底层都是 NumPy，例如字符串还受 StringDtype 的存储选择影响。

In [3]:
raw = pd.DataFrame({"qty": [2, None, 5], "ready": [True, None, False]}, index=["A", "B", "C"])
nullable = raw.convert_dtypes(dtype_backend="numpy_nullable")
arrow_backed = raw.convert_dtypes(dtype_backend="pyarrow")
print(nullable.dtypes)  # 预期：qty 为 Int64，ready 为 boolean。
print(arrow_backed.dtypes)  # 预期：qty 为 int64[pyarrow]，ready 为 bool[pyarrow]。
print(arrow_backed)
print(nullable.isna().equals(arrow_backed.isna()))
# qty 分别是 Int64 与 int64[pyarrow]；ready 分别是 boolean 与 bool[pyarrow]。
# 缺失位置相同，索引仍为 A、B、C；原始 raw 不会被原地转换。

qty        Int64
ready    boolean
dtype: object
qty      int64[pyarrow]
ready     bool[pyarrow]
dtype: object
    qty  ready
A     2   True
B  <NA>   <NA>
C     5  False
True


## 4 解析引擎与存储后端

read_csv 的 engine 决定由谁解析 CSV；dtype_backend 决定返回列采用哪类类型。两者可以独立选择：用 C 引擎也能得到 ArrowDtype，用 PyArrow 引擎也能返回 pandas 可空类型。

以下继续读取 csv_text，每次使用新的 StringIO，避免读取位置影响比较。这里只比较结果，不测速度，也不根据引擎名称预定性能结论。

In [4]:
for engine in ["c", "pyarrow"]:
    for backend in ["numpy_nullable", "pyarrow"]:
        with StringIO(csv_text) as source:
            result = pd.read_csv(source, engine=engine, dtype_backend=backend)
        # 预期：numpy_nullable 对应 string/Int64，pyarrow 对应 string[pyarrow]/int64[pyarrow]；两种引擎各显示一组。
        print(engine, backend, result["item"].dtype, result["qty"].dtype)
        print(result["qty"].isna().tolist())
        assert result["item"].tolist() == ["apple", "pear", "plum"]
# 两种引擎都能产生 Int64 或 int64[pyarrow]，缺失位置均为中间一行。

c numpy_nullable string Int64
[False, True, False]
c pyarrow string[pyarrow] int64[pyarrow]
[False, True, False]
pyarrow numpy_nullable string Int64
[False, True, False]
pyarrow pyarrow string[pyarrow] int64[pyarrow]
[False, True, False]


解析引擎支持的参数并不相同。本环境的 PyArrow CSV 引擎不支持 chunksize；如果需要逐块读取，又想使用 Arrow 列，可以选择 C 引擎并保留 dtype_backend="pyarrow"。

In [5]:
# 预期 ValueError：本环境的 PyArrow CSV 引擎不支持 chunksize 分块参数。
with StringIO(csv_text) as source:
    pd.read_csv(source, engine="pyarrow", chunksize=2)

ValueError: The 'chunksize' option is not supported with the 'pyarrow' engine

In [6]:
with StringIO(csv_text) as source:
    with pd.read_csv(source, engine="c", dtype_backend="pyarrow", chunksize=2) as reader:
        for chunk in reader:
            print(chunk["item"].tolist(), chunk["qty"].dtype)
# 两块分别含 apple、pear 和 plum；qty 仍为 int64[pyarrow]。

['apple', 'pear'] int64[pyarrow]
['plum'] int64[pyarrow]


## 5 字符串的类型与存储

pandas 3 默认推断字符串为 str，使用 NaN 表示缺失；已安装 PyArrow 时，默认底层使用 Arrow。显式 string 则使用 pd.NA，底层存储受字符串存储设置影响，不能仅从 "string" 推断。

"string[pyarrow]" 是使用 Arrow 存储的 StringDtype，仍不等于 pd.ArrowDtype(pa.string())。区分它们要同时看 dtype 类、缺失语义与操作结果。

| 写法 | 中文名称／含义 | 缺失标记 |
| --- | --- | --- |
| str | pandas 3 默认字符串类型，属于 StringDtype | NaN |
| string | 可空字符串类型，属于 StringDtype | pd.NA |
| string[pyarrow] | 明确使用 Arrow 存储的 StringDtype | pd.NA |
| pd.ArrowDtype(pa.string()) | Arrow 字符串类型，属于 ArrowDtype | pd.NA |

In [7]:
text_values = ["apple", None, "pear"]
default_text = pd.Series(text_values)
nullable_text = pd.Series(text_values, dtype="string")
arrow_string = pd.Series(text_values, dtype="string[pyarrow]")
arrow_dtype_text = pd.Series(text_values, dtype=pd.ArrowDtype(pa.string()))
for label, values in [
    ("default", default_text), ("string", nullable_text),
    ("string[pyarrow]", arrow_string), ("ArrowDtype", arrow_dtype_text),
]:
    # 预期：前三项为 StringDtype，最后为 ArrowDtype；本环境的 storage 均为 pyarrow，默认字符串显示 str。
    print(label, type(values.dtype).__name__, values.dtype, values.dtype.storage)
    print(values.tolist())
print(arrow_string.dtype == arrow_dtype_text.dtype)
# 此环境四者都使用 pyarrow 存储；只有 default 的缺失显示为 nan。
# 最后的类型比较为 False，不能因底层同为 Arrow 就认为 dtype 相同。

default StringDtype str pyarrow
['apple', nan, 'pear']
string StringDtype string pyarrow
['apple', <NA>, 'pear']
string[pyarrow] StringDtype string pyarrow
['apple', <NA>, 'pear']
ArrowDtype ArrowDtype string[pyarrow] pyarrow
['apple', <NA>, 'pear']
False


## 6 操作结果也有类型

Arrow 支持数值运算、比较、聚合和许多字符串操作，但不同类型的结果不必相同。str.contains 检查是否含有文本；下面用 regex=False 明确进行字面匹配。

默认 str 的 contains 在缺失处默认为 False；可空 StringDtype 保留 pd.NA，结果为 boolean；ArrowDtype 字符串保留 pd.NA，结果为 bool[pyarrow]。需要把缺失统一视为未匹配时，可以显式传 na=False。

In [8]:
for label, values in [
    ("default", default_text), ("StringDtype", arrow_string),
    ("ArrowDtype", arrow_dtype_text),
]:
    matched = values.str.contains("a", regex=False)
    print(label, matched.tolist(), matched.dtype)
print(arrow_dtype_text.str.contains("a", regex=False, na=False))
# 三组匹配值在缺失位置的处理不同；显式 na=False 后中间一行为 False。

default [True, False, True] bool
StringDtype [True, <NA>, True] boolean
ArrowDtype [True, <NA>, True] bool[pyarrow]
0     True
1    False
2     True
dtype: bool[pyarrow]


继续使用 counts，数值计算和比较会保留缺失；字符串 upper 将文本转成大写。支持范围还取决于具体 dtype、pandas 和 PyArrow 版本，不能从这些例子推断所有 pandas 操作都由 Arrow 实现或都不会复制。

In [9]:
print(counts + 1)  # 预期：A=3、B=<NA>、C=6，dtype 为 int64[pyarrow]。
print(counts > 2)  # 预期：A=False、B=<NA>、C=True，dtype 为 bool[pyarrow]。
print(counts.mean())
print(arrow_dtype_text.str.upper())
# 加法结果为 3、<NA>、6；比较结果为 False、<NA>、True。
# 均值只使用两个非缺失值，得到 3.5；字符串缺失仍保留。

A       3
B    <NA>
C       6
dtype: int64[pyarrow]
A    False
B     <NA>
C     True
dtype: bool[pyarrow]
3.5
0    APPLE
1     <NA>
2     PEAR
dtype: string[pyarrow]


字符串类型只接收字符串或缺失值，底层使用 Arrow 并不意味着能像 object 一样混放任意 Python 对象。下面尝试把数值写入默认 str 列，观察它被拒绝。

In [10]:
checked_text = default_text.copy()

# 预期 TypeError：默认 str 列不接受整数 10，只能写入字符串或缺失值。
checked_text.iloc[0] = 10

TypeError: Invalid value '10' for dtype 'str'. Value should be a string or missing value, got 'int' instead.

In [11]:
print(checked_text.tolist())
# 原来的 apple、缺失、pear 保留；如业务要文本编号，应先明确转换成字符串。

['apple', nan, 'pear']


## 7 转成 NumPy 时会改变什么

Series.to_numpy 返回没有 pandas 行标签的 ndarray。扩展类型可能转换为不同的 NumPy dtype，缺失标记也可能改变；copy=False 只是不强制复制，不保证零复制。

下面继续使用带缺失的 counts。默认结果以浮点数和 NaN 表达整数及缺失；需要保留 Python 整数与 pd.NA 时，可以明确选择 object。两种结果都不再携带 A、B、C 标签。

In [12]:
numeric_values = counts.to_numpy()
object_values = counts.to_numpy(dtype=object)
print(numeric_values, numeric_values.dtype)  # 预期：[2. nan 5.] float64。
print(object_values, object_values.dtype)
print(counts.index.tolist())
# 默认是 [2. nan 5.]、float64；object 结果为 [2 <NA> 5]。
# 要交给依赖位置的计算时，应另行保留索引并确认对应顺序。

[ 2. nan  5.] float64
[2 <NA> 5] object
['A', 'B', 'C']


NumPy 普通整数 dtype 不能表示 pd.NA。若下游必须接收整数数组，先制定缺失策略：保留单独的掩码，再选业务不会混淆的替代值。这里约定合法数量非负，用 -1 表示缺失，不能把这个约定套用到任意数据。

In [13]:
# 预期 TypeError：当前列含 pd.NA，不能直接转换为不支持该缺失值的 NumPy int64 数组。
counts.to_numpy(dtype="int64")

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NAType'

In [14]:
missing_mask = counts.isna().to_numpy()
integer_values = counts.to_numpy(dtype="int64", na_value=-1)
print(integer_values, integer_values.dtype)
print(missing_mask)
# 整数结果为 [2 -1 5]，掩码为 [False True False]；缺失位置可独立追踪。

[ 2 -1  5] int64
[False  True False]


copy=True 可以要求独立的 NumPy 数组。对于后续需要原地修改的数值计算，明确复制比猜测底层是否共享更直接。下面使用无缺失整数，修改输出不会改变原 Series。

In [15]:
complete = pd.Series([2, 5], dtype="int64[pyarrow]")
owned = complete.to_numpy(copy=True)
owned[0] = 99
print(owned)
print(complete.tolist())
# 独立数组为 [99 5]；原数据仍为 [2, 5]。

[99  5]
[2, 5]


## 8 选学：Arrow Table 与类型往返
Arrow Table 是由带名称的列组成的表。to_pandas 默认选择兼容的 pandas 表示，不表示所有列都继续使用 ArrowDtype；传入 types_mapper=pd.ArrowDtype 可明确要求 Arrow 支持的列类型。

下面直接从 Arrow 构造表，它没有 pandas 来源的类型元数据。带缺失的整数默认回到 float64，指定映射后保持为 int64[pyarrow]。

In [16]:
plain_table = pa.table({"qty": [2, None, 5]})
ordinary_frame = plain_table.to_pandas()
arrow_frame = plain_table.to_pandas(types_mapper=pd.ArrowDtype)
print(ordinary_frame)  # 预期：qty 为 2.0、NaN、5.0，缺失值迫使这组普通 NumPy 整数转为浮点。
print(ordinary_frame.dtypes)  # 预期：qty 为 float64。
print(arrow_frame)
print(arrow_frame.dtypes)
# 行顺序和值的含义相同，但 dtype 与缺失标记不同。

   qty
0  2.0
1  NaN
2  5.0
qty    float64
dtype: object
    qty
0     2
1  <NA>
2     5
qty    int64[pyarrow]
dtype: object


## 9 选学：索引元数据
Table.from_pandas 默认用 schema 中的 pandas 元数据记录索引信息。普通索引通常成为额外的物理列，to_pandas 再根据元数据恢复；preserve_index=False 则不保存索引。

以下原表使用 pandas 可空整数 Int64。来源元数据也有助于恢复这种类型，因此它与刚才没有 pandas 元数据的 plain_table 不同。

In [17]:
indexed = pd.DataFrame(
    {"qty": pd.array([2, None, 5], dtype="Int64")},
    index=pd.Index(["A", "B", "C"], name="row"),
)
table = pa.Table.from_pandas(indexed)
restored = table.to_pandas()
print(table.column_names)  # 预期：['qty', 'row']。
print(b"pandas" in table.schema.metadata)  # 预期：True。
print(restored)  # 预期：命名索引 row 的标签为 A/B/C，qty 为 2/<NA>/5。
print(restored.dtypes)  # 预期：qty 恢复为 Int64。
pd.testing.assert_frame_equal(indexed, restored)
without_index = pa.Table.from_pandas(indexed, preserve_index=False).to_pandas()
print(without_index.index.tolist())
# Table 物理列为 qty、row；默认往返恢复命名索引和 Int64。
# 不保存索引时读回的行号是 0、1、2，不再是 A、B、C。

['qty', 'row']
True
      qty
row      
A       2
B    <NA>
C       5
qty    Int64
dtype: object
[0, 1, 2]


RangeIndex 默认只写入元数据，不额外增加物理列；preserve_index=True 可强制把它也存成列。元数据若被忽略或丢弃，就不能假定原索引仍可恢复。

In [18]:
ranged = pd.DataFrame({"qty": [2, 5]}, index=pd.RangeIndex(10, 14, 2, name="row"))
range_table = pa.Table.from_pandas(ranged)
physical_table = pa.Table.from_pandas(ranged, preserve_index=True)
print(range_table.column_names, physical_table.column_names)  # 预期：['qty'] ['qty', 'row']，后者把索引写成物理列。
print(range_table.to_pandas().index)
print(range_table.to_pandas(ignore_metadata=True).index)
# 默认只有 qty 列，强制保存后增加 row 列；忽略元数据后索引退回 0、1。

['qty'] ['qty', 'row']
RangeIndex(start=10, stop=14, step=2, name='row')
RangeIndex(start=0, stop=2, step=1)


## 10 选学：零复制的具体条件
零复制指转换时共享已有的数据缓冲区，不重新复制该部分数据。对于 PyArrow Array 转 NumPy，无缺失、内存布局兼容的原始数值类型可返回视图；不能由此推广到所有 Arrow 类型或整张表。

Array.to_numpy(zero_copy_only=True) 在必须复制时抛出异常。成功的共享视图只读，因为 Arrow 数据不可变；需要可写数组时允许复制并指定 writable=True。

In [19]:
arrow_numbers = pa.array([1, 2, 3], type=pa.int64())
view = arrow_numbers.to_numpy(zero_copy_only=True)
second_view = arrow_numbers.to_numpy(zero_copy_only=True)
print(view, view.flags.writeable)  # 预期：[1 2 3] False，只读视图。
print(np.shares_memory(view, second_view))

# 预期 ValueError：这个 NumPy 视图共享 Arrow 的只读数据，不能直接修改。
view[0] = 99

[1 2 3] False
True


ValueError: assignment destination is read-only

In [20]:
writable = arrow_numbers.to_numpy(zero_copy_only=False, writable=True)
writable[0] = 99
print(writable, arrow_numbers.to_pylist())
# 两个视图共享内存；修改可写副本后原 Arrow 数组仍为 [1, 2, 3]。

[99  2  3] [1, 2, 3]


含 null 的整数 Array 不能以这种方式零复制成为普通 NumPy 数组：缺失信息需要转换。允许复制后，此例得到带 NaN 的浮点数组。

In [21]:
arrow_missing = pa.array([1, None, 3], type=pa.int64())

# 预期 ArrowInvalid：含 null 的整数 Arrow 数组需要转换缺失表示，无法满足 zero_copy_only=True。
arrow_missing.to_numpy(zero_copy_only=True)

ArrowInvalid: Needed to copy 1 chunks with 1 nulls, but zero_copy_only was True

In [22]:
converted = arrow_missing.to_numpy(zero_copy_only=False)
print(converted, converted.dtype)
# 结果为 [1. nan 3.]，dtype 为 float64。

[ 1. nan  3.] float64


多个 Arrow 片段也不能直接组成一个连续的 NumPy 视图。PyArrow 25 的 ChunkedArray.to_numpy 要求 zero_copy_only=False；它与单个 Array 的接口条件不同。下面只核对这个两片段数值例，不把结论扩写成 Arrow 后端的所有转换都会复制。

In [23]:
chunks = pa.chunked_array([[1, 2], [3]], type=pa.int64())

# 预期 ValueError：ChunkedArray.to_numpy 要求 zero_copy_only=False，不能直接请求零复制视图。
chunks.to_numpy(zero_copy_only=True)

ValueError: zero_copy_only must be False for pyarrow.ChunkedArray.to_numpy

In [24]:
joined = chunks.to_numpy(zero_copy_only=False)
first_chunk_view = chunks.chunk(0).to_numpy(zero_copy_only=True)
print(chunks.num_chunks, joined)
print(np.shares_memory(joined, first_chunk_view))
# 两个片段合成 [1 2 3]；此结果与第一个原片段不共享内存。

2 [1 2 3]
False


## 本章小结

（1）ArrowDtype 描述列类型，dtype_backend 选择返回列的类型体系；CSV engine 另行决定解析方式。

（2）底层同为 Arrow，不代表字符串 dtype、缺失语义和操作结果相同。检查类型类及实际结果，不只看名称中的 pyarrow。

（3）转换为 NumPy 会去掉标签，可能改变 dtype、缺失表示并分配新内存；copy=False 不保证零复制。

（4）Arrow Table 往返需约定索引元数据与目标类型；共享条件取决于具体接口、数据布局、缺失值和片段数。

## 练习

（1）读取下面的自制 CSV，要求数量保留 Arrow 可空整数。列出数量缺失的标签，计算已知数量总和，并核对三行的顺序与 dtype。

In [25]:
practice_csv = "label,qty\nA,3\nB,\nC,4\n"
# 补充：使用 StringIO 管理输入，给出 dtype_backend，打印表和 dtypes。
# 检查：缺失标签为 B，已知总量为 7；说明这不代表全部库存的确定总量。

（2）先预测以下两种字符串的类型是否相同、匹配结果的 dtype 和缺失值，再运行。说明为何都使用 Arrow 存储却得到不同结果。

In [26]:
first_text = pd.Series(["cat", None], dtype="string[pyarrow]")
second_text = pd.Series(["cat", None], dtype=pd.ArrowDtype(pa.string()))
print(first_text.dtype == second_text.dtype)
print(first_text.str.contains("c", regex=False))
print(second_text.str.contains("c", regex=False))
# 补充：记录预测，解释 StringDtype 与 ArrowDtype 的区别。

False
0    True
1    <NA>
dtype: boolean
0    True
1    <NA>
dtype: bool[pyarrow]


（3）原任务一次读取完整 CSV，现在新增约束：必须每次最多读取 2 行，但结果仍要使用 ArrowDtype。你会怎样选择 engine 与 dtype_backend？说明理由并实现。不要把所有块收集成一个完整表。

In [27]:
chunk_csv = "label,qty\nA,1\nB,2\nC,3\n"
# 补充：使用合适的解析引擎，逐块累计数量，及时关闭 reader 和输入流。
# 检查：每块最多 2 行、qty 为 int64[pyarrow]，合计为 6。

（4）下游只接收可写的 NumPy 整数数组，原输入却含缺失值。请说明为何不能同时直接保留 pd.NA 并承诺零复制，制定缺失表示和复制策略，再检查修改输出不会影响原 Series。

In [28]:
input_counts = pd.Series([4, None, 7], index=["A", "B", "C"], dtype="int64[pyarrow]")
# 约定：有效数量均非负，允许使用 -1 作为缺失替代值，并保留单独掩码。
# 补充：明确 dtype、na_value、copy；保存标签，修改结果中的第一个值。
# 检查：结果可写，原 A 仍为 4，原 B 仍缺失；解释选择理由。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | 主指南与读取 API 核查版本为 pandas 3.0.6。[PyArrow Functionality](https://pandas.pydata.org/docs/user_guide/pyarrow.html) 的 Data Structure Integration、StringDtype 与 ArrowDtype 对比、Operations、I/O Reading；[字符串迁移指南](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的默认 str、NaN、底层 Arrow、非字符串赋值与可空 string；[ArrowDtype](https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html)、[StringDtype](https://pandas.pydata.org/docs/reference/api/pandas.StringDtype.html) 的类型及 storage、na_value（这两页缓存标识为 3.0.5，示例按 3.0.6 执行核对）；[convert_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html) 的 dtype_backend；[read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) 的 engine、dtype_backend、chunksize 与 [IO tools：Specifying the parser engine](https://pandas.pydata.org/docs/user_guide/io.html#specifying-the-parser-engine) 的 PyArrow 参数限制；[str.contains](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) 的 na、regex；[Series.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_numpy.html) 的 dtype、na_value、copy 和扩展类型转换。 |
| Apache Arrow 官方文档 | Apache Arrow 25.0.1 [Pandas Integration](https://arrow.apache.org/docs/python/pandas.html) 的 DataFrames、Handling pandas Indexes、Nullable types、Memory Usage and Zero Copy；[Table.from_pandas / to_pandas](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html) 的 preserve_index、ignore_metadata、types_mapper；[Array.to_numpy](https://arrow.apache.org/docs/python/generated/pyarrow.Array.html#pyarrow.Array.to_numpy) 的 zero_copy_only、writable 与 null 限制；[ChunkedArray.to_numpy](https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html#pyarrow.ChunkedArray.to_numpy) 的连续数组与复制要求。指南中的旧 pandas 类型限制不作为 pandas 3 的统一规则，本章采用当前 API 条件和实测 dtype。 |
| NumPy 官方文档 | NumPy 2.5 [shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html)：判断两个数组是否共享内存；仅用于本章小型数值数组。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[pyarrow](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/pyarrow.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[io](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/io.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |